In [41]:
%load_ext cudf.pandas
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential, layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping

The cudf.pandas extension is already loaded. To reload it, use:
  %reload_ext cudf.pandas


In [42]:
# importing weather forcasting dataset

zip_path = tf.keras.utils.get_file(
    origin="https://storage.googleapis.com/tensorflow/tf-keras-datasets/jena_climate_2009_2016.csv.zip",
    fname="jena_climate_2009_2016.csv.zip",
    extract=True,
)

In [43]:
zip_path

'/root/.keras/datasets/jena_climate_2009_2016_extracted'

In [44]:
import os
csv_path = os.path.join(zip_path, "jena_climate_2009_2016.csv")

df = pd.read_csv(csv_path)
df.head()

,Date Time,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),rho (g/m**3),wv (m/s),max. wv (m/s),wd (deg)
0,01.01.2009 00:10:00,996.52,-8.02,265.40,-8.90,93.3,3.33,3.11,0.22,1.94,3.12,1307.75,1.03,1.75,152.3
1,01.01.2009 00:20:00,996.57,-8.41,265.01,-9.28,93.4,3.23,3.02,0.21,1.89,3.03,1309.80,0.72,1.50,136.1
2,01.01.2009 00:30:00,996.53,-8.51,264.91,-9.31,93.9,3.21,3.01,0.20,1.88,3.02,1310.24,0.19,0.63,171.6
3,01.01.2009 00:40:00,996.51,-8.31,265.12,-9.07,94.2,3.26,3.07,0.19,1.92,3.08,1309.19,0.34,0.50,198.0
4,01.01.2009 00:50:00,996.51,-8.27,265.15,-9.04,94.1,3.27,3.08,0.19,1.92,3.09,1309.00,0.32,0.63,214.3


In [45]:
df.columns

Index(['Date Time', 'p (mbar)', 'T (degC)', 'Tpot (K)', 'Tdew (degC)',
       'rh (%)', 'VPmax (mbar)', 'VPact (mbar)', 'VPdef (mbar)', 'sh (g/kg)',
       'H2OC (mmol/mol)', 'rho (g/m**3)', 'wv (m/s)', 'max. wv (m/s)',
       'wd (deg)'],
      dtype='object')

In [46]:
column_mapping = {
    "Date Time": "date_time",
    "p (mbar)": "pressure",
    "T (degC)": "temperature",
    "Tpot (K)": "potential_temp",
    "Tdew (degC)": "dew_point",
    "rh (%)": "relative_humidity",
    "VPmax (mbar)": "max_vapor_pressure",
    "VPact (mbar)": "actual_vapor_pressure",
    "VPdef (mbar)": "vapor_pressure_deficit",
    "sh (g/kg)": "specific_humidity",
    "H2OC (mmol/mol)": "water_vapor_concentration",
    "rho (g/m**3)": "air_density",
    "wv (m/s)": "wind_speed",
    "max. wv (m/s)": "max_wind_speed",
    "wd (deg)": "wind_direction",
}

df.rename(columns=column_mapping, inplace=True)

with pd.option_context('display.max_columns',None):
    display(df.head())

,date_time,pressure,temperature,potential_temp,dew_point,relative_humidity,max_vapor_pressure,actual_vapor_pressure,vapor_pressure_deficit,specific_humidity,water_vapor_concentration,air_density,wind_speed,max_wind_speed,wind_direction
0,01.01.2009 00:10:00,996.52,-8.02,265.40,-8.90,93.3,3.33,3.11,0.22,1.94,3.12,1307.75,1.03,1.75,152.3
1,01.01.2009 00:20:00,996.57,-8.41,265.01,-9.28,93.4,3.23,3.02,0.21,1.89,3.03,1309.80,0.72,1.50,136.1
2,01.01.2009 00:30:00,996.53,-8.51,264.91,-9.31,93.9,3.21,3.01,0.20,1.88,3.02,1310.24,0.19,0.63,171.6
3,01.01.2009 00:40:00,996.51,-8.31,265.12,-9.07,94.2,3.26,3.07,0.19,1.92,3.08,1309.19,0.34,0.50,198.0
4,01.01.2009 00:50:00,996.51,-8.27,265.15,-9.04,94.1,3.27,3.08,0.19,1.92,3.09,1309.00,0.32,0.63,214.3


In [47]:
print(df.shape)

df.describe()

(420551, 15)


,pressure,temperature,potential_temp,dew_point,relative_humidity,max_vapor_pressure,actual_vapor_pressure,vapor_pressure_deficit,specific_humidity,water_vapor_concentration,air_density,wind_speed,max_wind_speed,wind_direction
count,420551.000000,420551.000000,420551.000000,420551.000000,420551.000000,420551.000000,420551.000000,420551.000000,420551.000000,420551.000000,420551.000000,420551.000000,420551.000000,420551.000000
mean,989.212776,9.450147,283.492743,4.955854,76.008259,13.576251,9.533756,4.042412,6.022408,9.640223,1216.062748,1.702224,3.056555,174.743738
std,8.358481,8.423365,8.504471,6.730674,16.476175,7.739020,4.184164,4.896851,2.656139,4.235395,39.975208,65.446714,69.016932,86.681693
min,913.600000,-23.010000,250.600000,-25.010000,12.950000,0.950000,0.790000,0.000000,0.500000,0.800000,1059.450000,-9999.000000,-9999.000000,0.000000
25%,984.200000,3.360000,277.430000,0.240000,65.210000,7.780000,6.210000,0.870000,3.920000,6.290000,1187.490000,0.990000,1.760000,124.900000
50%,989.580000,9.420000,283.470000,5.220000,79.300000,11.820000,8.860000,2.190000,5.590000,8.960000,1213.790000,1.760000,2.960000,198.100000
75%,994.720000,15.470000,289.530000,10.070000,89.400000,17.600000,12.350000,5.300000,7.800000,12.490000,1242.770000,2.860000,4.740000,234.100000
max,1015.350000,37.280000,311.340000,23.110000,100.000000,63.770000,28.320000,46.010000,18.130000,28.820000,1393.540000,28.490000,23.500000,360.000000


In [48]:
df.info()

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 420551 entries, 0 to 420550
Data columns (total 15 columns):
 #   Column                     Non-Null Count   Dtype
---  ------                     --------------   -----
 0   date_time                  420551 non-null  object
 1   pressure                   420551 non-null  float64
 2   temperature                420551 non-null  float64
 3   potential_temp             420551 non-null  float64
 4   dew_point                  420551 non-null  float64
 5   relative_humidity          420551 non-null  float64
 6   max_vapor_pressure         420551 non-null  float64
 7   actual_vapor_pressure      420551 non-null  float64
 8   vapor_pressure_deficit     420551 non-null  float64
 9   specific_humidity          420551 non-null  float64
 10  water_vapor_concentration  420551 non-null  float64
 11  air_density                420551 non-null  float64
 12  wind_speed                 420551 non-null  float64
 13  max_wind_speed             42055

In [49]:
df['date_time'] = pd.to_datetime(df['date_time'], format='%d.%m.%Y %H:%M:%S')

In [50]:
df['date_time']

0        2009-01-01 00:10:00
1        2009-01-01 00:20:00
2        2009-01-01 00:30:00
3        2009-01-01 00:40:00
4        2009-01-01 00:50:00
                 ...        
420546   2016-12-31 23:20:00
420547   2016-12-31 23:30:00
420548   2016-12-31 23:40:00
420549   2016-12-31 23:50:00
420550   2017-01-01 00:00:00
Name: date_time, Length: 420551, dtype: datetime64[ns]

In [51]:
df

,date_time,pressure,temperature,potential_temp,dew_point,relative_humidity,max_vapor_pressure,actual_vapor_pressure,vapor_pressure_deficit,specific_humidity,water_vapor_concentration,air_density,wind_speed,max_wind_speed,wind_direction
0,2009-01-01 00:10:00,996.52,-8.02,265.40,-8.90,93.30,3.33,3.11,0.22,1.94,3.12,1307.75,1.03,1.75,152.3
1,2009-01-01 00:20:00,996.57,-8.41,265.01,-9.28,93.40,3.23,3.02,0.21,1.89,3.03,1309.80,0.72,1.50,136.1
2,2009-01-01 00:30:00,996.53,-8.51,264.91,-9.31,93.90,3.21,3.01,0.20,1.88,3.02,1310.24,0.19,0.63,171.6
3,2009-01-01 00:40:00,996.51,-8.31,265.12,-9.07,94.20,3.26,3.07,0.19,1.92,3.08,1309.19,0.34,0.50,198.0
4,2009-01-01 00:50:00,996.51,-8.27,265.15,-9.04,94.10,3.27,3.08,0.19,1.92,3.09,1309.00,0.32,0.63,214.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
420546,2016-12-31 23:20:00,1000.07,-4.05,269.10,-8.13,73.10,4.52,3.30,1.22,2.06,3.30,1292.98,0.67,1.52,240.0
420547,2016-12-31 23:30:00,999.93,-3.35,269.81,-8.06,69.71,4.77,3.32,1.44,2.07,3.32,1289.44,1.14,1.92,234.3
420548,2016-12-31 23:40:00,999.82,-3.16,270.01,-8.21,67.91,4.84,3.28,1.55,2.05,3.28,1288.39,1.08,2.00,215.2
420549,2016-12-31 23:50:00,999.81,-4.23,268.94,-8.53,71.80,4.46,3.20,1.26,1.99,3.20,1293.56,1.49,2.16,225.8


In [52]:
# hourly data 
hourly_df = df[5::6].copy()
hourly_df.set_index('date_time', inplace=True)

In [53]:
target_col_index = hourly_df.columns.get_loc('temperature')        # returns the values in array format, so no need to convert it into array
data = hourly_df.values

##### In RNN based models, you cant just use `num / 255.0`, it will throw error. Hence follow the code below

In [54]:
split_data = int(len(data) * 0.8)
mean = data[ : split_data].mean(axis=0)
std = data[ : split_data].std(axis=0)

scaled_data = (data - mean) / std

##### In time series based data, we cant use `train_test_splilt`, coz it messes up the sequence of the series. Hence below code is recommeded

In [55]:
time_steps = 10

x_seq, y_seq = [], []

for i in range(len(scaled_data) - time_steps):
    x_seq.append(scaled_data[i : i + time_steps])
    y_seq.append(data[i + time_steps, target_col_index])

x_seq = np.array(x_seq)
y_seq = np.array(y_seq)

# splitting the data : 80% for training, and 20% for testing
split = int(len(x_seq) * 0.8)

x_train = x_seq[ : split]
y_train = y_seq[ : split]
x_test = x_seq[split : ]
y_test = y_seq[split : ]

print(x_train.shape)
print(x_test.shape)

(56064, 10, 14)
(14017, 10, 14)


In [56]:
import optuna

In [57]:
def my_model(trial):
    lr = trial.suggest_float('lr', 1e-8, 1e-2, log=True)
    n_layers = trial.suggest_int('n_layers', 1,5, step=1)

    time_steps = 10
    
    model = keras.Sequential([
        keras.Input(shape=(time_steps, 14))                                             # RNN requires a 3D tensor of shape (samples, time_steps, features) — specifically (samples, 10, 14)
    ])

    for i in range(n_layers):
        neurons = trial.suggest_int(f'neurons_{i}', 1,200, step=16)
        activation_fn = trial.suggest_categorical(f'activation_fn_{i}', ['relu', 'leaky_relu', 'elu', 'tanh'])
        dropouts = trial.suggest_float(f'dropouts_{i}', 0.0,0.5, step=0.1)
        
        initializer = trial.suggest_categorical(f'initializer_{i}', ['he_normal', 'glorot_normal'])
        regularizer = trial.suggest_categorical(f'regularizer_{i}', ['l1','l2'])
        reg_rate = trial.suggest_float(f'reg_rate_{i}', 1e-8, 1e-2, log=True)
        if regularizer == 'l1':
            chosen_regularizer = regularizers.l1(reg_rate)
        else:
            chosen_regularizer = regularizers.l2(reg_rate)

        bias_initializer = trial.suggest_categorical(f'bias_initializer_{i}', ['he_normal', 'glorot_normal', 'he_uniform', 'glorot_uniform'])
        bias_regularizer = trial.suggest_categorical(f'bias_regularizer_{i}', ['l1','l2'])
        bias_reg_rate = trial.suggest_float(f'bias_reg_rate_{i}', 1e-8, 1e-2, log=True)

        if bias_regularizer == 'l1':
            chosen_bias_regularizer = regularizers.l1(bias_reg_rate)
        else:
            chosen_bias_regularizer = regularizers.l2(bias_reg_rate)
        

        if i < (n_layers -1):
            return_sequence = True
        else:
            return_sequence = False

        model.add(layers.SimpleRNN(
            neurons, 
            return_sequences = return_sequence,
            activation = activation_fn,
            kernel_initializer=initializer, 
            kernel_regularizer=chosen_regularizer, 
            bias_initializer=bias_initializer,
            bias_regularizer=chosen_bias_regularizer            
        ))

        if dropouts > 0.0:
            model.add(layers.Dropout(dropouts))

    neurons = trial.suggest_int(f'ANN_neurons', 1,200, step=16)
    activation_fn = trial.suggest_categorical(f'ANN_activation_fn', ['relu', 'leaky_relu', 'elu', 'tanh'])
    model.add(layers.Dense(
        neurons,
        activation = activation_fn        
    ))
    
    model.add(layers.Dense(1))                 # using softmax is not recommended, only for the classification problem


    model.compile(
        loss = keras.losses.MeanSquaredError(),
        optimizer = keras.optimizers.Adam(lr),
        metrics = ['mae']    
    )
    

    early_stopping = EarlyStopping(
        monitor = 'val_loss',
        patience = 3,
        restore_best_weights = True,
        verbose = 1
    )


    history = model.fit(
        x_train, y_train,
        epochs = 5,
        batch_size = 256,
        validation_data = (x_test, y_test),
        callbacks = [early_stopping],
        shuffle = False,                      # for time series data you must  expicitly set "shuffle = False"
        verbose = 1
    )

    model.save(f"model_trial_{trial.number}.keras")

    return min(history.history['val_loss'])


study = optuna.create_study(direction='minimize')
study.optimize(my_model, n_trials = 10)


[I 2026-09-28 08:24:09,042] A new study created in memory with name: no-name-6b5664d5-07d2-4f1c-be38-b903222f9bca
/tmp/ipykernel_6641/2171826843.py:12: UserWarning: The distribution is specified by [1, 200] and step=16, but the range is not divisible by `step`. It will be replaced with [1, 193].
  neurons = trial.suggest_int(f'neurons_{i}', 1,200, step=16)
/tmp/ipykernel_6641/2171826843.py:52: UserWarning: The distribution is specified by [1, 200] and step=16, but the range is not divisible by `step`. It will be replaced with [1, 193].
  neurons = trial.suggest_int(f'ANN_neurons', 1,200, step=16)


Epoch 1/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 14s 20ms/step - loss: 95.6528 - mae: 7.1884 - val_loss: 54.0441 - val_mae: 4.8660
Epoch 2/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 37.2217 - mae: 3.8232 - val_loss: 31.3882 - val_mae: 3.1726
Epoch 3/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 25.6069 - mae: 2.8490 - val_loss: 24.2214 - val_mae: 2.5880
Epoch 4/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 20.7618 - mae: 2.3711 - val_loss: 19.8997 - val_mae: 2.1056
Epoch 5/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 18.3331 - mae: 2.1111 - val_loss: 17.2175 - val_mae: 1.7478
Restoring model weights from the end of the best epoch: 5.


[I 2026-09-28 08:24:31,067] Trial 0 finished with value: 17.21754264831543 and parameters: {'lr': 8.325521010955104e-05, 'n_layers': 5, 'neurons_0': 81, 'activation_fn_0': 'relu', 'dropouts_0': 0.1, 'initializer_0': 'he_normal', 'regularizer_0': 'l1', 'reg_rate_0': 0.005822554641659796, 'bias_initializer_0': 'he_uniform', 'bias_regularizer_0': 'l2', 'bias_reg_rate_0': 1.7219609552498425e-08, 'neurons_1': 113, 'activation_fn_1': 'leaky_relu', 'dropouts_1': 0.0, 'initializer_1': 'he_normal', 'regularizer_1': 'l1', 'reg_rate_1': 0.007030695056645348, 'bias_initializer_1': 'glorot_normal', 'bias_regularizer_1': 'l2', 'bias_reg_rate_1': 5.02802020612274e-07, 'neurons_2': 65, 'activation_fn_2': 'elu', 'dropouts_2': 0.4, 'initializer_2': 'he_normal', 'regularizer_2': 'l2', 'reg_rate_2': 0.0016800702693478004, 'bias_initializer_2': 'glorot_uniform', 'bias_regularizer_2': 'l2', 'bias_reg_rate_2': 1.8351389279616213e-05, 'neurons_3': 129, 'activation_fn_3': 'relu', 'dropouts_3': 0.0, 'initialize

Epoch 1/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step - loss: 118.0938 - mae: 8.9080 - val_loss: 84.9908 - val_mae: 7.5322
Epoch 2/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 32.3470 - mae: 4.2453 - val_loss: 14.0106 - val_mae: 2.6612
Epoch 3/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 18.2285 - mae: 3.1840 - val_loss: 9.3272 - val_mae: 2.1618
Epoch 4/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 14.8879 - mae: 2.8700 - val_loss: 7.2122 - val_mae: 1.9203
Epoch 5/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 12.6833 - mae: 2.6562 - val_loss: 6.2027 - val_mae: 1.7885
Restoring model weights from the end of the best epoch: 5.


[I 2026-09-28 08:24:44,058] Trial 1 finished with value: 6.202683448791504 and parameters: {'lr': 2.622900482083332e-05, 'n_layers': 2, 'neurons_0': 113, 'activation_fn_0': 'tanh', 'dropouts_0': 0.4, 'initializer_0': 'glorot_normal', 'regularizer_0': 'l1', 'reg_rate_0': 6.376829673593783e-05, 'bias_initializer_0': 'he_uniform', 'bias_regularizer_0': 'l1', 'bias_reg_rate_0': 2.8494467122803826e-08, 'neurons_1': 65, 'activation_fn_1': 'leaky_relu', 'dropouts_1': 0.5, 'initializer_1': 'he_normal', 'regularizer_1': 'l2', 'reg_rate_1': 3.165244978800842e-08, 'bias_initializer_1': 'glorot_uniform', 'bias_regularizer_1': 'l1', 'bias_reg_rate_1': 5.4374576367990246e-05, 'ANN_neurons': 177, 'ANN_activation_fn': 'leaky_relu'}. Best is trial 1 with value: 6.202683448791504.


...

[I 2026-09-28 08:26:20,147] Trial 7 finished with value: 180.38575744628906 and parameters: {'lr': 5.603402328117463e-08, 'n_layers': 2, 'neurons_0': 49, 'activation_fn_0': 'leaky_relu', 'dropouts_0': 0.5, 'initializer_0': 'glorot_normal', 'regularizer_0': 'l2', 'reg_rate_0': 2.331911645536913e-07, 'bias_initializer_0': 'glorot_uniform', 'bias_regularizer_0': 'l1', 'bias_reg_rate_0': 1.1698720165001146e-05, 'neurons_1': 65, 'activation_fn_1': 'elu', 'dropouts_1': 0.0, 'initializer_1': 'glorot_normal', 'regularizer_1': 'l2', 'reg_rate_1': 0.00019503257971128813, 'bias_initializer_1': 'glorot_uniform', 'bias_regularizer_1': 'l2', 'bias_reg_rate_1': 2.0123286675537028e-08, 'ANN_neurons': 113, 'ANN_activation_fn': 'tanh'}. Best is trial 1 with value: 6.202683448791504.


Epoch 1/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - loss: 17.3735 - mae: 2.7853 - val_loss: 13.9229 - val_mae: 2.8636
Epoch 2/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 4.3344 - mae: 1.4189 - val_loss: 5.0885 - val_mae: 1.5719
Epoch 3/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 3.3977 - mae: 1.2325 - val_loss: 6.8261 - val_mae: 1.8840
Epoch 4/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 3.0466 - mae: 1.1493 - val_loss: 5.0239 - val_mae: 1.5905
Epoch 5/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 2.5464 - mae: 1.0183 - val_loss: 4.2294 - val_mae: 1.4232
Restoring model weights from the end of the best epoch: 5.


[I 2026-09-28 08:26:32,207] Trial 8 finished with value: 4.2293500900268555 and parameters: {'lr': 0.00043120967906937945, 'n_layers': 2, 'neurons_0': 129, 'activation_fn_0': 'tanh', 'dropouts_0': 0.0, 'initializer_0': 'he_normal', 'regularizer_0': 'l1', 'reg_rate_0': 2.2927988176426535e-07, 'bias_initializer_0': 'glorot_uniform', 'bias_regularizer_0': 'l2', 'bias_reg_rate_0': 4.610723081637976e-08, 'neurons_1': 113, 'activation_fn_1': 'relu', 'dropouts_1': 0.1, 'initializer_1': 'he_normal', 'regularizer_1': 'l2', 'reg_rate_1': 0.0033140426876145618, 'bias_initializer_1': 'he_normal', 'bias_regularizer_1': 'l1', 'bias_reg_rate_1': 1.5424625919566547e-06, 'ANN_neurons': 97, 'ANN_activation_fn': 'elu'}. Best is trial 8 with value: 4.2293500900268555.


Epoch 1/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 15s 20ms/step - loss: 131.6955 - mae: 9.3909 - val_loss: 165.7837 - val_mae: 10.5428
Epoch 2/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 131.1048 - mae: 9.3690 - val_loss: 165.1460 - val_mae: 10.5204
Epoch 3/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 130.6279 - mae: 9.3526 - val_loss: 164.5080 - val_mae: 10.4978
Epoch 4/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - loss: 130.5401 - mae: 9.3462 - val_loss: 163.8664 - val_mae: 10.4752
Epoch 5/5
219/219 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 130.1761 - mae: 9.3266 - val_loss: 163.2270 - val_mae: 10.4526
Restoring model weights from the end of the best epoch: 5.


[I 2026-09-28 08:26:55,482] Trial 9 finished with value: 163.2269744873047 and parameters: {'lr': 4.745795877986555e-08, 'n_layers': 5, 'neurons_0': 177, 'activation_fn_0': 'tanh', 'dropouts_0': 0.1, 'initializer_0': 'he_normal', 'regularizer_0': 'l2', 'reg_rate_0': 6.837289969738668e-06, 'bias_initializer_0': 'he_normal', 'bias_regularizer_0': 'l1', 'bias_reg_rate_0': 1.8882017907974023e-05, 'neurons_1': 113, 'activation_fn_1': 'relu', 'dropouts_1': 0.0, 'initializer_1': 'he_normal', 'regularizer_1': 'l2', 'reg_rate_1': 0.0004370652606243078, 'bias_initializer_1': 'he_uniform', 'bias_regularizer_1': 'l1', 'bias_reg_rate_1': 9.023781683065826e-08, 'neurons_2': 17, 'activation_fn_2': 'relu', 'dropouts_2': 0.1, 'initializer_2': 'he_normal', 'regularizer_2': 'l2', 'reg_rate_2': 1.4667698022720416e-07, 'bias_initializer_2': 'glorot_normal', 'bias_regularizer_2': 'l2', 'bias_reg_rate_2': 9.935242410691244e-05, 'neurons_3': 161, 'activation_fn_3': 'elu', 'dropouts_3': 0.1, 'initializer_3': '

In [63]:
print(study.best_trial.number)
print()
study.best_params

8



{'lr': 0.00043120967906937945,
 'n_layers': 2,
 'neurons_0': 129,
 'activation_fn_0': 'tanh',
 'dropouts_0': 0.0,
 'initializer_0': 'he_normal',
 'regularizer_0': 'l1',
 'reg_rate_0': 2.2927988176426535e-07,
 'bias_initializer_0': 'glorot_uniform',
 'bias_regularizer_0': 'l2',
 'bias_reg_rate_0': 4.610723081637976e-08,
 'neurons_1': 113,
 'activation_fn_1': 'relu',
 'dropouts_1': 0.1,
 'initializer_1': 'he_normal',
 'regularizer_1': 'l2',
 'reg_rate_1': 0.0033140426876145618,
 'bias_initializer_1': 'he_normal',
 'bias_regularizer_1': 'l1',
 'bias_reg_rate_1': 1.5424625919566547e-06,
 'ANN_neurons': 97,
 'ANN_activation_fn': 'elu'}

In [64]:
num = study.best_trial.number
model = keras.models.load_model(f"model_trial_{num}.keras")
model.summary()

Model: "sequential_18"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_52 (SimpleRNN)       │ (None, 10, 129)        │        18,576 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_53 (SimpleRNN)       │ (None, 113)            │        27,459 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_43 (Dropout)            │ (None, 113)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_36 (Dense)                │ (None, 97)             │        11,058 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_37 (Dense)                │ (None, 1)              │            98 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 171,575 (670.22 KB)

 Trainable params: 57,191 (223.40 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 114,384 (446.82 KB)

In [72]:
model.fit(
    x_train, y_train,
    epochs = 50,
    batch_size=256,
    shuffle=False
)

Epoch 1/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.4015 - mae: 0.4450
Epoch 2/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.3842 - mae: 0.4309
Epoch 3/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.4195 - mae: 0.4541
Epoch 4/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.4619 - mae: 0.4781
Epoch 5/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.4707 - mae: 0.4807
Epoch 6/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.4313 - mae: 0.4623
Epoch 7/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.4211 - mae: 0.4536
Epoch 8/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.3871 - mae: 0.4324
Epoch 9/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.4295 - mae: 0.4622
Epoch 10/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.4954 - mae: 0.4983
Epoch 11/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.3893 - mae: 0.4342
Epoch 12/50
219/219 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.3857 - mae: 0.4338
Epoch 13/50
219/219 ━━━━━

**IMP :** above the `model.fit()` was trained multiple times, to reduce the error form 2.0545 to 0.39 / 0.40

In [73]:
pred = model.predict(x_test)

439/439 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step


In [74]:

test_loss, test_mae = model.evaluate(x_test, y_test, verbose=1)

print(f"\\nTest Loss (MSE): {test_loss:.4f}")
print(f"Test MAE: {test_mae:.4f} °C")


print("\\n\n\n--- Sample Predictions vs Actual Temperatures ---")
for i in range(10):
    pred_temp = pred[i][0]
    actual_temp = y_test[i]
    difference = abs(pred_temp - actual_temp)
    
    print(f"Predicted: {pred_temp:5.2f}°C  |  Actual: {actual_temp:5.2f}°C  |  Off by: {difference:4.2f}°C")

439/439 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 1.7859 - mae: 0.9971
\nTest Loss (MSE): 1.7859
Test MAE: 0.9971 °C
\n

--- Sample Predictions vs Actual Temperatures ---
Predicted: 14.63°C  |  Actual: 15.82°C  |  Off by: 1.19°C
Predicted: 13.86°C  |  Actual: 15.02°C  |  Off by: 1.16°C
Predicted: 13.48°C  |  Actual: 14.14°C  |  Off by: 0.66°C
Predicted: 12.79°C  |  Actual: 13.26°C  |  Off by: 0.47°C
Predicted: 11.73°C  |  Actual: 12.77°C  |  Off by: 1.04°C
Predicted: 12.38°C  |  Actual: 12.62°C  |  Off by: 0.24°C
Predicted: 11.69°C  |  Actual: 12.48°C  |  Off by: 0.79°C
Predicted: 12.44°C  |  Actual: 12.85°C  |  Off by: 0.41°C
Predicted: 12.45°C  |  Actual: 12.43°C  |  Off by: 0.02°C
Predicted: 12.03°C  |  Actual: 12.63°C  |  Off by: 0.60°C
